# 04 - Send the same run to Agent 365 and Azure

**Goal:** keep the working agent and Azure trace viewer from notebook 02, and add the real Agent 365 exporter. These are two separate destinations for the same run.

**Before starting:** finish [00 Identity](./00_Identity_Setup.ipynb), [01 Permissions](./01_Agent_Permissions.ipynb), and [02 Agent and Azure traces](./02_Agent_and_Azure_Traces.ipynb). Notebook 03 (OBO) is not required. **Restart the kernel.**

Think of Azure tracing as the worker's detailed receipt. Agent 365 gives the organization its own management/security view. Seeing the receipt in Azure does not prove that Agent 365 has received its copy.

**No pretend success:** creating an identity, receiving a token, flushing a queue, and seeing a local table are each useful, but none alone proves Agent 365 ingestion.

## 1. Prepare the Agent 365 tenant - once, administrator

1. Confirm at least one tenant user has an **assigned Microsoft 365 E7 or Microsoft Agent 365 license**. Token success alone does not meet the ingestion prerequisite.
2. Choose the route for the **existing child**:
   - **Registered Agent 365 agent:** an authorized administrator completes Agent 365 onboarding and confirms the registered instance is this child. Microsoft's registered-agent S2S route does not require an explicit observability role grant.
   - **Existing Entra-only child:** explicitly grant telemetry-write permission using step 2 below. This enables telemetry authorization; it is **not full Agent 365 registration** and does not guarantee an agent management card.
3. Have the administrator confirm which Agent 365 activity/monitoring surface is available to this tenant and grant the presenter access **before the demo**. Do not promise a specific trace viewer without checking it.

**If you need full Agent 365 onboarding:** use [Microsoft's registration guide](https://learn.microsoft.com/microsoft-agent-365/developer/get-started). The Agent 365 CLI registration-only retry requires an already configured CLI project; full setup can create additional objects/resources. Do not run full setup blindly over an existing manual identity.

**Stop here if licensing, authorization, or the tenant viewer is unavailable.** Notebook 02 still provides a valid Azure tracing demo, but label Agent 365 integration as not yet verified.

## 2. Entra-only route: grant telemetry write - once, administrator

**Skip this step if** your administrator has confirmed the child is Agent 365 registered with the supported S2S route.

Otherwise, use PowerShell 7 with Microsoft.Graph.Applications and an authorized consent administrator. Replace the tenant and child **Object ID**. The fixed API ID belongs to Microsoft, not to this workshop.

```powershell
$ErrorActionPreference = 'Stop'
Connect-MgGraph -TenantId '<TENANT-ID>' -Scopes 'Application.Read.All','AppRoleAssignment.ReadWrite.All' -UseDeviceAuthentication -ContextScope Process -ErrorAction Stop
$childObjectId = '<CHILD-OBJECT-ID>'
$apis = @(Get-MgServicePrincipal -Filter "appId eq '9b975845-388f-4429-889e-eab1ef63949c'" -Property Id,AppRoles)
if ($apis.Count -ne 1) { throw 'Administrator must establish the official Agent 365 service principal first.' }
$api = $apis[0]
$roles = @($api.AppRoles | Where-Object { $_.Value -eq 'Agent365.Observability.OtelWrite' -and $_.IsEnabled -and $_.AllowedMemberTypes -contains 'Application' })
if ($roles.Count -ne 1) { throw 'Expected one enabled telemetry-write application role.' }
$role = $roles[0]
$existing = @(Get-MgServicePrincipalAppRoleAssignment -ServicePrincipalId $childObjectId -All | Where-Object { $_.ResourceId -eq $api.Id -and $_.AppRoleId -eq $role.Id })
if ($existing.Count -eq 0) {
    New-MgServicePrincipalAppRoleAssignment -ServicePrincipalId $childObjectId -PrincipalId $childObjectId -ResourceId $api.Id -AppRoleId $role.Id
}
```

Allow propagation and obtain a fresh token. Do not grant only the parent blueprint and assume the child has access.

[Microsoft's authentication and permission routes](https://learn.microsoft.com/microsoft-agent-365/developer/observability-authentication-setup)

## 3. Load the same agent - each fresh kernel

**Setup needed:** steps 1-2 as applicable, all settings/roles from notebook 02, current CLI sign-in, and the project kernel. Start from the project root or `Agent_365`. Do not rerun tracing setup in a used kernel.

We use the same child for MCP and Agent 365, but request **different tokens**, each addressed to the correct API.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == 'Agent_365' else Path.cwd()
sys.path.insert(0, str(root))
from Agent_365.agent365_auth import BlueprintTokenResolver
from Agent_365.mcp_workshop import MCPSettings, sign_in_agent, run_mcp_assistant
from Agent_365.workshop_helpers import start_recording, show_activities
from Agent_365.workshop_tracing import configure_workshop_tracing, flush_workshop_trace
from microsoft.opentelemetry.a365.core import BaggageBuilder

settings = MCPSettings.from_env()

## 4. Authenticate telemetry and turn on both destinations

**Setup needed:** step 3. No additional registration is done by this cell.

The blueprint credential obtains the child's token for `api://9b975845-388f-4429-889e-eab1ef63949c/.default`. The SDK chooses Microsoft's exporter endpoint. Passing `token_resolver` below enables Agent 365 **in addition to** Azure export.

Azure ingestion still uses your CLI user's Monitoring Metrics Publisher role. The Agent 365 exporter uses the child identity. We never reuse the MCP token for telemetry.

In [ ]:
token_resolver = BlueprintTokenResolver(
    tenant_id=settings.tenant_id,
    agent_id=settings.agent_id,
    blueprint_id=settings.blueprint_id,
    blueprint_secret=settings.blueprint_secret,
)
await token_resolver.sign_in()
configure_workshop_tracing(
    settings.tenant_id, settings.agent_id, token_resolver=token_resolver
)
access = await sign_in_agent(settings)

## 5. Run the same protected-tool agent

**Setup needed:** step 4 succeeded. The baggage context is like a luggage tag: it carries the tenant and child ID through nested spans. It is metadata, not an access grant.

Agent Framework records the real invocation, model calls, and tool calls. Both exporters receive that run; the local table is a third, in-memory copy.

In [ ]:
start_recording()
with BaggageBuilder().tenant_id(settings.tenant_id).agent_id(settings.agent_id).build():
    await run_mcp_assistant(settings, access)
show_activities()
trace_id = await flush_workshop_trace()

## 6. Verify each destination separately

**Setup needed:** query/view permissions and the tenant viewer confirmed in step 1. Use the run's trace ID, child ID, and run time to correlate available records.

| Place | What to check | What it proves |
|---|---|---|
| Application Insights > Logs | Run the printed exact-trace query; inspect agent/model/tool spans | Azure received operational traces |
| Tenant's available Agent 365 monitoring surface | Administrator locates corresponding agent activity; Microsoft 365 admin center/Defender/Purview offer different views | Agent 365 received and surfaced activity, only once actually found |
| Entra > Agents > child > Sign-in logs | Authentication events for the child | Identity authentication, NOT execution traces |

**Do not mark Agent 365 verified just because Azure has rows.** The tenant may expose summaries/security records rather than a raw OpenTelemetry trace tree. If the administrator cannot locate corresponding activity, report Agent 365 ingestion/display as **unverified**.

**If export fails:** HTTP 401 means check token identity/audience/expiry; HTTP 403 or `insufficient_scope` means check registration or the child's telemetry-write role; accepted requests without visible activity still require checking assigned licensing, tenant capabilities, delay, and viewing permissions.

**Say:** "Identity is the badge, MCP enforces the key, Azure shows the execution receipt, and Agent 365 has its own onboarding and tenant-side verification. These are connected, but they are not the same thing."

Clear outputs before sharing; revoke temporary grants and credentials after the workshop. [Back to the workshop map](./WORKSHOP_START_HERE.md)